# Practice Activity: Weather as a Business Covariate

**GSB 5544 · Computing and Machine Learning for Business Analytics**

You manage inventory planning for a beverage distributor with two markets: **San Luis Obispo, CA** and **Phoenix, AZ**. Before anyone builds a demand model, you need each market's weather history, and it lives in a bucket that is far larger than your laptop.

**Run this notebook inside your SageMaker Studio JupyterLab space.** Every code cell begins with a `# RUNS ON:` line that says where it executes. Everything you need is in the pre-class reading and the topics-of-practice notebook. Each question has a code cell; some also ask for a short written answer in a markdown cell. Q1 and Q11 need numbers you wrote down from the reading.

**Stations**

| Market | Station ID | Name |
|---|---|---|
| San Luis Obispo | `USC00047851` | SAN LUIS OBISPO POLY, CA |
| Phoenix | `USW00023183` | PHOENIX AIRPORT, AZ |

Run the two setup cells first.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
%pip install -q s3fs             # harmless if already present

import io
import json
import time
import datetime as dt
import psutil
import boto3
import pandas as pd
import matplotlib.pyplot as plt
from botocore import UNSIGNED
from botocore.config import Config

BUCKET   = "noaa-ghcn-pds"
REGION   = "us-east-1"
COLS     = ["id", "date", "element", "value", "m_flag", "q_flag", "s_flag", "obs_time"]
STATIONS = {"San Luis Obispo": "USC00047851", "Phoenix": "USW00023183"}

# unsigned: the public bucket does not care who is asking
s3 = boto3.client("s3", region_name=REGION, config=Config(signature_version=UNSIGNED))


def human(n, base=1000):
    for unit in ["B", "KB", "MB", "GB", "TB", "PB"]:
        if n < base:
            return f"{n:,.1f} {unit}"
        n /= base
    return f"{n:,.1f} EB"

**Athena setup** (given; it is block J of the topics-of-practice notebook, unchanged). It creates your results bucket if needed, defines the helpers, and defines the `gsb5544.ghcn` table over the public Parquet files. DDL is free.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
sts     = boto3.client("sts",    region_name=REGION)
athena  = boto3.client("athena", region_name=REGION)
s3_mine = boto3.client("s3",     region_name=REGION)

ACCOUNT        = sts.get_caller_identity()["Account"]
RESULTS_BUCKET = f"gsb5544-athena-{ACCOUNT}"
RESULTS        = f"s3://{RESULTS_BUCKET}/athena/"
s3_mine.create_bucket(Bucket=RESULTS_BUCKET)

ATHENA_BYTES = 0                                  # running total of everything scanned today; Q13 audits it

def run_athena(sql, database=None):
    """Start a query, wait for it, return (query id, statistics)."""
    global ATHENA_BYTES
    kwargs = {"QueryString": sql, "ResultConfiguration": {"OutputLocation": RESULTS}}
    if database:
        kwargs["QueryExecutionContext"] = {"Database": database}
    qid = athena.start_query_execution(**kwargs)["QueryExecutionId"]
    while True:
        q = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]
        state = q["Status"]["State"]
        if state in ("SUCCEEDED", "FAILED", "CANCELLED"):
            break
        time.sleep(1)
    if state != "SUCCEEDED":
        raise RuntimeError(q["Status"].get("StateChangeReason", state))
    ATHENA_BYTES += q["Statistics"].get("DataScannedInBytes", 0)
    return qid, q["Statistics"]


def athena_df(qid):
    """Read a finished query's result CSV from the results bucket."""
    obj = s3_mine.get_object(Bucket=RESULTS_BUCKET, Key=f"athena/{qid}.csv")
    return pd.read_csv(io.BytesIO(obj["Body"].read()))


run_athena("CREATE DATABASE IF NOT EXISTS gsb5544")
run_athena('''
CREATE EXTERNAL TABLE IF NOT EXISTS gsb5544.ghcn (
    id string, `date` string, data_value bigint,
    m_flag string, q_flag string, s_flag string, obs_time string
)
PARTITIONED BY (year int, element string)
STORED AS PARQUET
LOCATION 's3://noaa-ghcn-pds/parquet/by_year/'
TBLPROPERTIES (
    'projection.enabled'        = 'true',
    'projection.year.type'      = 'integer',
    'projection.year.range'     = '1750,2030',
    'projection.element.type'   = 'enum',
    'projection.element.values' = 'TMAX,TMIN,PRCP,SNOW,SNWD,TAVG',
    'storage.location.template' = 's3://noaa-ghcn-pds/parquet/by_year/YEAR=${year}/ELEMENT=${element}/'
)
''', database="gsb5544")
print("Ready. Results bucket:", RESULTS_BUCKET)

## Set 1 · Capacity

### Q1. Two machines
Report physical cores, total RAM, available RAM, and free disk in GB for **this** machine (the SageMaker instance) and for **your laptop** (the numbers you wrote down from the reading, §1), as one DataFrame with a row per measure and a column per machine. Store this machine's available RAM in bytes as `MY_RAM`.

Then, in the markdown cell, one sentence: which machine would you rather use to read a 1.3 GB file from `noaa-ghcn-pds`, and why?

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
vm = psutil.virtual_memory()
du = psutil.disk_usage("/")
MY_RAM = vm.available

machines = pd.DataFrame({
    "laptop": {                      # from the reading, §1
        "physical cores":     0,
        "total RAM (GB)":     0.0,
        "available RAM (GB)": 0.0,
        "free disk (GB)":     0.0,
    },
    "SageMaker ml.t3.medium": {
        "physical cores":     psutil.cpu_count(logical=False),
        "total RAM (GB)":     round(vm.total / 1e9, 1),
        "available RAM (GB)": round(vm.available / 1e9, 1),
        "free disk (GB)":     round(du.free / 1e9, 1),
    },
})
machines

*Your answer:*

### Q2. Rows that fit
A GHCN row is roughly 50 bytes on disk, and a text-heavy CSV needs about 3× its disk size once loaded in pandas. How many raw rows could this machine's available RAM hold? Print the number with thousands separators.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
BYTES_PER_ROW = 50
MULT = 3
rows_fit = MY_RAM // (BYTES_PER_ROW * MULT)
print(f"About {rows_fit:,} rows fit in {human(MY_RAM)} of available RAM on this instance")

### Q3. A decade of weather
List the by-year files for 2016 through 2025 and compute their **total** size. Print the total in readable units, then print how many copies of this machine's available RAM that is.

*Hint: a `Prefix` of `csv/by_year/201` returns 2010–2019; you will need two listings or a filter.*

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
total = 0
for prefix in ["csv/by_year/201", "csv/by_year/202"]:
    resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix)
    for obj in resp["Contents"]:
        year = int(obj["Key"].split("/")[-1][:4])
        if 2016 <= year <= 2025:
            total += obj["Size"]
            print(f"{obj['Key']:<28} {human(obj['Size'])}")

print()
print("Total 2016–2025 :", human(total))
print("Copies of my RAM:", f"{total / MY_RAM:.1f}×")

## Set 2 · Vocabulary in action

### Q4. What is under `csv/`?
List the common prefixes directly under `csv/` in the bucket. Then, in the markdown cell that follows, explain in one or two sentences why `by_year/` is not a folder and what S3 is actually doing when it shows you one.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
resp = s3.list_objects_v2(Bucket=BUCKET, Prefix="csv/", Delimiter="/")
print([p["Prefix"] for p in resp.get("CommonPrefixes", [])])

*Your answer:*

### Q5. Metadata only
Using a single call that does **not** download the file, print the size (readable) and the last-modified date of `csv/by_year/2024.csv`. Then print whether a plain `pd.read_csv` of it would fit in this machine's RAM at 3×.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
meta = s3.head_object(Bucket=BUCKET, Key="csv/by_year/2024.csv")
size = meta["ContentLength"]
print("Size         :", human(size))
print("Last modified:", meta["LastModified"])
print("Fits at 3×?  :", size * 3 < MY_RAM)

## Set 3 · Extraction

### Q6. Pull both markets
Write a function `load_station(station_id)` that:

1. prints the object's size before reading it,
2. checks whether the file has a header row using a range request,
3. reads the file keeping only `id`, `date`, `element`, `value`,
4. converts `date` to a datetime,
5. keeps only `TMAX` and `PRCP` rows from **2015 onward**,
6. returns the DataFrame.

Call it for both stations and combine the results into one DataFrame called `wx` with a `market` column.

This runs on the SageMaker instance: the bytes travel from S3 to this machine over AWS's network in the same region, and never touch your laptop.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
def load_station(station_id):
    key = f"csv/by_station/{station_id}.csv"
    print(station_id, "size:", human(s3.head_object(Bucket=BUCKET, Key=key)["ContentLength"]))

    first = s3.get_object(Bucket=BUCKET, Key=key, Range="bytes=0-99")["Body"].read().decode()
    has_header = first.upper().startswith("ID,")

    raw = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()
    df = pd.read_csv(
        io.BytesIO(raw),
        header=0 if has_header else None,
        names=COLS,
        usecols=["id", "date", "element", "value"],
        dtype={"id": "string", "element": "string"},
    )
    df["date"] = pd.to_datetime(df["date"].astype(str), format="%Y%m%d")
    df = df[df["element"].isin(["TMAX", "PRCP"]) & (df["date"] >= "2015-01-01")]
    return df.reset_index(drop=True)


frames = []
for market, sid in STATIONS.items():
    d = load_station(sid)
    d["market"] = market
    frames.append(d)

wx = pd.concat(frames, ignore_index=True)
print(wx.shape)
wx.groupby(["market", "element"]).size()

### Q7. Hot days per year
Temperatures are stored in tenths of a degree Celsius. For each market and year, count the number of days with a daily maximum of **35 °C or higher** (95 °F). Produce a table with years as rows and markets as columns.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
tmax = wx[wx["element"] == "TMAX"].copy()
tmax["tmax_c"] = tmax["value"] / 10
tmax["year"] = tmax["date"].dt.year

hot = (
    tmax[tmax["tmax_c"] >= 35]
    .groupby(["year", "market"]).size()
    .unstack("market")
    .fillna(0).astype(int)
)
hot

### Q8. Monthly rainfall
Precipitation is stored in tenths of a millimeter. For each market, compute total precipitation per month in millimeters, then show the **average for each calendar month** across all years (so 12 rows per market). Which month is wettest in each market?

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
prcp = wx[wx["element"] == "PRCP"].copy()
prcp["prcp_mm"] = prcp["value"] / 10

monthly_total = (
    prcp.set_index("date")
        .groupby("market")["prcp_mm"]
        .resample("ME").sum()           # "M" on pandas < 2.2
        .reset_index()
)
monthly_total["month"] = monthly_total["date"].dt.month

climatology = monthly_total.groupby(["market", "month"])["prcp_mm"].mean().unstack("market").round(1)
print(climatology)
print()
print("Wettest month:", climatology.idxmax().to_dict())

## Set 4 · Visualize and interpret

### Q9. One chart, two markets
Plot hot days per year (from Q7) for both markets on a single line chart. Label the axes with units and give the chart a title that says what it shows.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
fig, ax = plt.subplots(figsize=(9, 4))
for market in hot.columns:
    ax.plot(hot.index, hot[market], marker="o", linewidth=1.3, label=market)
ax.set_xlabel("Year")
ax.set_ylabel("Days with max temperature ≥ 35 °C")
ax.set_title("Hot days per year, 2015 onward")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

### Q10. The business read
In five to seven sentences, answer as the inventory planner:

- What does the chart imply about how differently the two markets should be stocked across the year?
- Name **two** other datasets you would want to join to this weather history before trusting a demand model, and say where each one probably lives (your laptop, a company database, or a cloud bucket).
- This weather history will be refreshed **every week** once the model is live. Of the three places this module's code can run (your laptop, a SageMaker instance, the Athena fleet), which would you choose for that weekly refresh, and why? Answer from the reading now; revise after Q11 if the numbers change your mind.

*Your answer:*

## Set 5 · Where the work runs

### Q11. Three ways to get the same rows
Extract the **Phoenix `TMAX` and `PRCP` rows for 2024** three ways and time each one:

- **(a) Laptop.** You already did this before class: the timed chunked read of `csv/by_year/2024.csv` in the reading, §6. Paste your `LAPTOP_SECONDS` into the cell. (That read was for the San Luis Obispo station; same file, same amount of work.)
- **(b) SageMaker.** Run the same chunked read here, filtering each one-million-row chunk to the Phoenix station. Time it.
- **(c) Athena.** Query `gsb5544.ghcn` for the same rows. Time it and record the bytes scanned.

Build a DataFrame `race` with one row per tier and columns for seconds, bytes moved or scanned, and rows returned. Also confirm that (b) and (c) return the same row count as filtering `wx` to Phoenix and 2024.

In the markdown cell after, explain which tier you would use for this extraction and why, using the phrase "filter at the source." Say what changed between (a) and (b), and what changed between (b) and (c).

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
LAPTOP_SECONDS = 0.0             # <- paste from the reading, §6

key = "csv/by_year/2024.csv"
by_year_bytes = s3.head_object(Bucket=BUCKET, Key=key)["ContentLength"]
first = s3.get_object(Bucket=BUCKET, Key=key, Range="bytes=0-99")["Body"].read().decode()
has_header = first.upper().startswith("ID,")

t0 = time.perf_counter()
chunks = pd.read_csv(
    f"s3://{BUCKET}/{key}",
    storage_options={"anon": True},
    header=0 if has_header else None,
    names=COLS,
    usecols=["id", "date", "element", "value"],
    dtype={"id": "string", "element": "string"},
    chunksize=1_000_000,
)
sm_rows = pd.concat(c[c["id"] == STATIONS["Phoenix"]] for c in chunks)
sm_rows = sm_rows[sm_rows["element"].isin(["TMAX", "PRCP"])]
SAGEMAKER_SECONDS = time.perf_counter() - t0
print(f"SageMaker: {len(sm_rows):,} rows in {SAGEMAKER_SECONDS:,.0f} s, read {human(by_year_bytes)}")

In [ ]:
# RUNS ON: Athena fleet (results land in S3)
SQL = f'''
SELECT id, "date", data_value
FROM   gsb5544.ghcn
WHERE  year = 2024
  AND  element IN ('TMAX', 'PRCP')
  AND  id = '{STATIONS["Phoenix"]}'
'''

t0 = time.perf_counter()
qid, stats = run_athena(SQL, database="gsb5544")
athena_rows = athena_df(qid)
ATHENA_SECONDS = time.perf_counter() - t0
athena_bytes = stats["DataScannedInBytes"]

by_station = wx[(wx["market"] == "Phoenix") & (wx["date"].dt.year == 2024)]

race = pd.DataFrame(
    {
        "seconds":                [LAPTOP_SECONDS, SAGEMAKER_SECONDS, ATHENA_SECONDS],
        "bytes moved or scanned": [by_year_bytes, by_year_bytes, athena_bytes],
        "rows returned":          [None, len(sm_rows), len(athena_rows)],
    },
    index=["laptop", "SageMaker ml.t3.medium", "Athena fleet"],
)
race["readable"] = race["bytes moved or scanned"].map(human)
race["seconds"] = race["seconds"].round(1)

print(f"by_station path from Q6: {len(by_station):,} rows (read {human(sum(s3.head_object(Bucket=BUCKET, Key=f'csv/by_station/{sid}.csv')['ContentLength'] for sid in STATIONS.values()))} for both markets, all years)")
race

*Your answer:*

### Q12. Hot days, straight from the fleet
Write **one Athena query** that returns the number of hot days (`TMAX` ≥ 35 °C, i.e. `data_value >= 350`) per station per year from 2015 onward, for both stations: the same table you built in Q7, but computed entirely on the fleet. Pivot the result to years as rows and markets as columns and check it against `hot`.

Then print two numbers side by side: the bytes Athena scanned for this query, and the bytes pandas read in Q6 (the two by-station CSVs).

In the markdown cell, explain the comparison. One of the two paths read fewer bytes. Which, and why? What does that tell you about when Athena over `by_year` is the right tool, and when choosing the right object (the by-station file) beats it?

In [ ]:
# RUNS ON: Athena fleet (results land in S3)
SQL = f'''
SELECT id, year, COUNT(*) AS hot_days
FROM   gsb5544.ghcn
WHERE  element = 'TMAX'
  AND  year >= 2015
  AND  id IN ('{STATIONS["San Luis Obispo"]}', '{STATIONS["Phoenix"]}')
  AND  data_value >= 350
GROUP BY id, year
ORDER BY year, id
'''

qid, stats = run_athena(SQL, database="gsb5544")
id_to_market = {v: k for k, v in STATIONS.items()}

hot_athena = (
    athena_df(qid)
    .assign(market=lambda d: d["id"].map(id_to_market))
    .pivot(index="year", columns="market", values="hot_days")
    .fillna(0).astype(int)
)

athena_bytes = stats["DataScannedInBytes"]
pandas_bytes = sum(
    s3.head_object(Bucket=BUCKET, Key=f"csv/by_station/{sid}.csv")["ContentLength"]
    for sid in STATIONS.values()
)

print(f"Athena scanned : {human(athena_bytes):>9}   about ${max(athena_bytes, 10_000_000) / 1e12 * 5:.4f}")
print(f"pandas read    : {human(pandas_bytes):>9}   (the two by-station CSVs in Q6)")
same = (hot_athena == hot.reindex(hot_athena.index)[hot_athena.columns]).all().all()
print("Matches Q7     :", bool(same))
hot_athena

*Your answer:*

### Q13. Cost audit and teardown (graded)
Every session on a rented machine ends the same way. Four parts; the last one is the evidence.

**(a) Empty the results bucket.** Print the number of objects and their total size before deleting, then confirm the bucket is empty.

**(b) Estimate today's spend.** Hours this space has been running × the `ml.t3.medium` hourly price, plus `ATHENA_BYTES` × the Athena price per TB (10 MB minimum per query). Print the two lines and the total. The cell reads the space's start time from SageMaker; if that lookup fails, set `hours` by hand from the time you clicked **Run space** (Learner Lab: from the lab timer).

**(c) Stop the space.** In Studio → JupyterLab spaces, click **Stop** on `gsb5544` and wait for *Stopped*. Then read **Credits remaining** on Console Home (Learner Lab: the budget on the lab page).

**(d) Evidence.** Either paste a screenshot into the markdown cell showing the space *Stopped* and the credits readout, or run the verification cell from your **laptop** (it needs AWS credentials in `~/.aws/credentials`; Learner Lab users paste the "AWS Details" block there first) and paste its output into the markdown cell.

In [ ]:
# RUNS ON: SageMaker ml.t3.medium (us-east-1)
# (a) empty the results bucket
listing = s3_mine.list_objects_v2(Bucket=RESULTS_BUCKET)
objects = listing.get("Contents", [])
print(f"Results bucket : {len(objects)} objects, {human(sum(o['Size'] for o in objects))}")
if objects:
    s3_mine.delete_objects(Bucket=RESULTS_BUCKET, Delete={"Objects": [{"Key": o["Key"]} for o in objects]})
print("Bucket empty   :", "Contents" not in s3_mine.list_objects_v2(Bucket=RESULTS_BUCKET))

# (b) today's spend
INSTANCE_PER_HOUR = 0.05         # ml.t3.medium, us-east-1, approximate
ATHENA_PER_TB     = 5.00

try:
    meta = json.load(open("/opt/ml/metadata/resource-metadata.json"))
    sm = boto3.client("sagemaker", region_name=REGION)
    app = sm.describe_app(DomainId=meta["DomainId"], SpaceName=meta["SpaceName"],
                          AppType=meta["AppType"], AppName=meta["ResourceName"])
    hours = (dt.datetime.now(dt.timezone.utc) - app["CreationTime"]).total_seconds() / 3600
except Exception as e:
    print("Could not read the space's start time:", e)
    hours = 3.0                  # <- set by hand: hours since you clicked Run space

instance_cost = hours * INSTANCE_PER_HOUR
athena_cost   = max(ATHENA_BYTES, 10_000_000) / 1e12 * ATHENA_PER_TB
print(f"Instance       : {hours:.2f} h × ${INSTANCE_PER_HOUR}/h = ${instance_cost:.3f}")
print(f"Athena         : {human(ATHENA_BYTES)} scanned  = ${athena_cost:.4f}")
print(f"Total today    : ${instance_cost + athena_cost:.3f}")

**Verification cell** for part (d). This one runs on your laptop *after* you have stopped the space, because a cell cannot stop the machine it is running on and then report on it.

In [ ]:
# RUNS ON: laptop
# Needs AWS credentials in ~/.aws/credentials (Learner Lab: paste the "AWS Details" block there first).
import boto3

sm = boto3.client("sagemaker", region_name="us-east-1")
apps = sm.list_apps()["Apps"]
for a in apps:
    print(f'{a["AppType"]:<12} {a.get("SpaceName", ""):<12} {a["Status"]}')

still_running = [a for a in apps if a["Status"] not in ("Deleted", "Failed")]
print("All stopped" if not still_running else "STILL RUNNING: go back to Studio and stop it")

*Evidence (screenshot or pasted output):*

---

**Submit** this notebook with all cells run inside SageMaker, plus your Q13 evidence. Your written answers for Q1, Q4, Q10, Q11, and Q12 go in the markdown cells provided. A submission is not complete while its JupyterLab space is still running; we check.